Just want to get familiar with PyBoy

In [2]:
from pyboy import PyBoy 
import numpy as np
import time

In [ ]:
pyboy = PyBoy("../pokemon_rom.gbc")
for _ in range(60):
    pyboy.tick()

# save screen as PIL image
image = pyboy.screen.image
# image.save("screen.png")

# save as numpy array (144 x 160 x 4)
array = pyboy.screen.ndarray
print(array.shape)
pyboy.stop()


(144, 160, 4)


In [7]:
# read memory
memory = pyboy.memory[0xDCB8:0xDCB8+16]  # RAM
print(memory)  # print first 16 bytes of RAM   


[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [3]:
# run pyboy in sdl2 and check registers
pyboy = PyBoy("../pokemon_rom.gbc", window_type="SDL2")
for _ in range(60000):
    pyboy.tick(6, True, False)
    x = pyboy.memory[0xD20D]
    y = pyboy.memory[0xD20E]
    map_id = pyboy.memory[0xDA01]
    badges = pyboy.memory[0xD57C]
    in_battle = pyboy.memory[0xD116]
    hp = (pyboy.memory[0xDA4C] << 8) | pyboy.memory[0xDA4D]
    max_hp = (pyboy.memory[0xDA4E] << 8) | pyboy.memory[0xDA4F]
    
    print(f"X: {x}, Y: {y}, Map ID: {map_id}, Badges: {badges}, In Battle: {in_battle}, HP: {hp}, Max HP: {max_hp}")

pyboy.stop()
    

pyboy.pyboy                    ERROR    Deprecated use of 'window_type'. Use 'window' keyword argument instead. https://github.com/Baekalfen/PyBoy/wiki/Migrating-from-v1.x.x-to-v2.0.0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In Battle: 0, HP: 0, Max HP: 0
X: 0, Y: 0, Map ID: 0, Badges: 0, In

In [3]:
# Start PyBoy and render the SDL2 window while speeding up emulation
from pyboy import PyBoy

pyboy = PyBoy("../pokemon_rom.gbc", window="SDL2")
print("PyBoy started. Use the emulator window to choose your starter.")

try:
    while True:
        pyboy.tick(1.2, True, False)
except KeyboardInterrupt:
    print("Stopped ticking. The `pyboy` object is still available.")


PyBoy started. Use the emulator window to choose your starter.
Stopped ticking. The `pyboy` object is still available.


In [ ]:
# Save the current PyBoy state after you selected Totodile
import os

os.makedirs('../saves', exist_ok=True)

save_path = 'saves/totodile.state'

try:
    pyboy.save_state(save_path)
    print(f"Saved state to {save_path}")
except Exception:
    with open("../saves/totodile.state", "wb") as f:
        pyboy.save_state(f)
    print(f"Saved state to {save_path} (fallback)")


Saved state to saves/totodile.state (fallback)


: 

In [2]:
import time
from pyboy import PyBoy

save_path = '../saves/totodile.state'
pyboy2 = PyBoy("../pokemon_rom.gbc", window="SDL2")  # fixed: window= not window_type=

with open(save_path, "rb") as f:
    pyboy2.load_state(f)
print(f"State loaded from {save_path}")

for _ in range(2000):
    pyboy2.tick(2, True, False)

    # ── Global position (world coords — useful for debug, not used for tile tracking)
    x = pyboy2.memory[0xD20D]
    y = pyboy2.memory[0xD20E]

    # ── Tile exploration key: (map_bank, map_number, local_x, local_y)
    # map_bank + map_number together uniquely identify a map (map_number alone is not unique)
    map_bank   = pyboy2.memory[0xDA00]
    map_number = pyboy2.memory[0xDA01]
    local_x    = pyboy2.memory[0xDA02]
    local_y    = pyboy2.memory[0xDA03]

    # ── Badges: 0xD57C is a bitfield
    # bit 0 = Zephyr (Falkner), bit 1 = Hive, bit 2 = Plain, ... bit 7 = Rising
    badges      = pyboy2.memory[0xD57C]
    badge_count = bin(badges).count('1')
    zephyr      = bool(badges & 0x01)  # our win condition

    # ── Battle type: 0=overworld | 1=wild | 2=trainer | 3=gym  (VERIFY empirically)
    # Knowing the type lets the agent decide: run from wild battles to save steps,
    # fight trainer/gym battles for the large reward bonus.
    battle_type = pyboy2.memory[0xD116]

    # ── Party count
    party_count = pyboy2.memory[0xDA22]

    # ── Lead Pokemon HP — two separate addresses:
    #   DA4C/DA4D = party slot 0 HP  (always valid, persists between battles)
    #   CB1C/CB1D = active combat HP (only meaningful while battle_type > 0)
    lead_hp     = (pyboy2.memory[0xDA4C] << 8) | pyboy2.memory[0xDA4D]
    lead_max_hp = (pyboy2.memory[0xDA4E] << 8) | pyboy2.memory[0xDA4F]
    battle_hp   = (pyboy2.memory[0xCB1C] << 8) | pyboy2.memory[0xCB1D]
    hp_ratio    = lead_hp / lead_max_hp if lead_max_hp > 0 else 0.0

    # ── Event flags: raw bytes from the 0xD7B7–0xD8B6 "Flags in Game" block
    # Each byte holds 8 individual bit-flags. Watch which bit flips when you:
    #   - beat the rival in Cherrygrove     → flag_rival_cherrygrove
    #   - receive the egg from Mr. Pokemon  → flag_elm_mr_pokemon
    #   - enter Sprout Tower 2F / 3F        → flag_sprout_tower_2/3
    flag_rival_cherrygrove = pyboy2.memory[0xD8CA]  # "Met rival in Cherrygrove"
    flag_elm_mr_pokemon    = pyboy2.memory[0xD7BD]  # Elm quest / Mr. Pokemon discovery
    flag_sprout_tower_2    = pyboy2.memory[0xD85C]  # Sprout Tower 2F
    flag_sprout_tower_3    = pyboy2.memory[0xD85D]  # Sprout Tower 3F

    # ── Milestone maps: note the (map_bank, map_number) printed below when you
    # walk into: Cherrygrove | Mr. Pokemon's house (Rt30) | Violet City | Sprout Tower | Gym
    # Fill in MILESTONE_MAPS in env/ram_addresses.py once you have the values.

    print(
        f"TILE  bank={map_bank} map={map_number:3d} local=({local_x:3d},{local_y:3d})  "
        f"global=({x:3d},{y:3d})  |  "
        f"BATTLE type={battle_type}  PARTY {party_count}  "
        f"HP {lead_hp:3d}/{lead_max_hp:3d} ({hp_ratio:.0%}) bHP={battle_hp:3d}  |  "
        f"BADGES {badges:08b} cnt={badge_count} zephyr={zephyr}  |  "
        f"FLAGS rival={flag_rival_cherrygrove:08b} elm={flag_elm_mr_pokemon:08b} "
        f"sprout2={flag_sprout_tower_2:08b} sprout3={flag_sprout_tower_3:08b}"
    )
    time.sleep(0.05)

pyboy2.stop()
print("Done.")

State loaded from ../saves/totodile.state
TILE  bank=24 map=  4 local=(  8,  5)  global=(  9, 12)  |  BATTLE type=0  PARTY 1  HP  21/ 21 (100%) bHP=  0  |  BADGES 00000000 cnt=0 zephyr=False  |  FLAGS rival=10010111 elm=01000000 sprout2=00000000 sprout3=00000000
TILE  bank=24 map=  4 local=(  8,  5)  global=(  9, 12)  |  BATTLE type=0  PARTY 1  HP  21/ 21 (100%) bHP=  0  |  BADGES 00000000 cnt=0 zephyr=False  |  FLAGS rival=10010111 elm=01000000 sprout2=00000000 sprout3=00000000
TILE  bank=24 map=  4 local=(  8,  5)  global=(  9, 12)  |  BATTLE type=0  PARTY 1  HP  21/ 21 (100%) bHP=  0  |  BADGES 00000000 cnt=0 zephyr=False  |  FLAGS rival=10010111 elm=01000000 sprout2=00000000 sprout3=00000000
TILE  bank=24 map=  4 local=(  8,  5)  global=(  9, 12)  |  BATTLE type=0  PARTY 1  HP  21/ 21 (100%) bHP=  0  |  BADGES 00000000 cnt=0 zephyr=False  |  FLAGS rival=10010111 elm=01000000 sprout2=00000000 sprout3=00000000
TILE  bank=24 map=  4 local=(  8,  5)  global=(  9, 12)  |  BATTLE type=0 

In [2]:
from pyboy import PyBoy

pyboy3 = PyBoy("../pokemon_rom.gbc", window="SDL2", sound=False)
with open("../saves/totodile.state", "rb") as f:
    pyboy3.load_state(f)

WATCH = {
    "rival_cherrygrove": 0xD8CA,
    "elm_mr_pokemon":    0xD7BD,
    "sprout_tower_2":    0xD85C,
    "sprout_tower_3":    0xD85D,
    "badges":            0xD57C,
    "battle_type":       0xD116,
    "map_bank":          0xDA00,
    "map_number":        0xDA01,
}

prev = {name: pyboy3.memory[addr] for name, addr in WATCH.items()}
print("Watching for changes... (interrupt to stop)")
print(f"Initial state: {prev}\n")

try:
    while True:
        pyboy3.tick(2)
        for name, addr in WATCH.items():
            val = pyboy3.memory[addr]
            if val != prev[name]:
                print(f"  *** {name:25s}  {prev[name]:3d} (0b{prev[name]:08b})  →  {val:3d} (0b{val:08b})")
                prev[name] = val
except KeyboardInterrupt:
    pyboy3.stop()
    print("\nStopped.")

Watching for changes... (interrupt to stop)
Initial state: {'rival_cherrygrove': 151, 'elm_mr_pokemon': 64, 'sprout_tower_2': 0, 'sprout_tower_3': 0, 'badges': 0, 'battle_type': 0, 'map_bank': 24, 'map_number': 4}

  *** map_number                   4 (0b00000100)  →    3 (0b00000011)
  *** battle_type                  0 (0b00000000)  →    1 (0b00000001)
  *** battle_type                  1 (0b00000001)  →    0 (0b00000000)
  *** map_bank                    24 (0b00011000)  →   26 (0b00011010)
  *** map_number                   3 (0b00000011)  →    1 (0b00000001)
  *** battle_type                  0 (0b00000000)  →    1 (0b00000001)
  *** battle_type                  1 (0b00000001)  →    0 (0b00000000)
  *** battle_type                  0 (0b00000000)  →    1 (0b00000001)
  *** battle_type                  1 (0b00000001)  →    0 (0b00000000)
  *** map_number                   1 (0b00000001)  →   10 (0b00001010)
  *** map_number                  10 (0b00001010)  →    1 (0b00000001)
  **

: 

In [ ]:
pyboy.stop()

NameError: name 'pyboy' is not defined

: 